#1.What is Machine Learning

- Machine learning is the subset of artificial intelligence (AI) focused on algorithms that can “learn” the patterns of training data and, subsequently, make accurate inferences about new data. 
- This pattern recognition ability enables machine learning models to make decisions or predictions without explicit, hard-coded instructions.

#Types of Machine Learning
- Supervised Learning : Regression & Classification
- Unsupervised Learning: Clustering
- Reinforcement Learning

##Supervised Learning : Regression

In [0]:
pip install statsmodels

In [0]:
pip install openpyxl

In [0]:
import pandas as pd
import statsmodels.api as sm

In [0]:
df_oats = pd.read_excel("/Volumes/workspace/gettingstartedwithslr/gettingstartedwithslr_vol1/Oats_Yield_1866.xlsx")

In [0]:
df_oats.head()

In [0]:
df_oats.info()

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.scatter(df_oats["Year"], df_oats["Yield"], alpha=0.7)
plt.xlabel("Year")
plt.ylabel("Yield (bu/acre)")
plt.title("Oats Yield vs. Year")
plt.grid(True)
plt.show()

In [0]:
# Simple linear regression: yield_bu_acre (y) as a function of year (x)
X = sm.add_constant(df_oats[["Year"]])  # add a constant term for the intercept
y = df_oats["Yield"]

slr_model = sm.OLS(y, X).fit()
print(slr_model.summary())

Yield = (0.9476 * Year) - 1777.99

In [0]:
#2027 yield

y= (0.9467*2027) - 1777.99

In [0]:
y

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.scatter(df_oats["Year"], df_oats["Yield"], alpha=0.7)

plt.plot(df_oats["Year"], slr_model.predict(sm.add_constant(df_oats[["Year"]])), color="red", linewidth=2, label="SLR fit")

plt.legend()
plt.xlabel("Year")
plt.ylabel("Yield (bu/acre)")
plt.title("Oats Yield vs. Year")
plt.grid(True)
plt.show()

## Interpreting the OLS Regression Diagnostics — Healthy Ranges

| Diagnostic | What it tests | Healthy range / expected value | If outside the healthy range |
|---|---|---|---|
| **Omnibus** | Combined test for normality of residuals | Probability (Omnibus) **> 0.05** (test statistic itself should be **close to 0**) | Residuals are not normally distributed — consider transforming the target variable or inspecting outliers |
| **Skewness** | Symmetry of residual distribution | **Between -1 and +1** (ideally **close to 0**) | Skewed residuals — a non-linear transform (e.g. log) may help |
| **Kurtosis** | Tail heaviness of residual distribution | **Between -2 and +2** (ideally **close to 3** for raw kurtosis) | Heavy tails (outliers) or light tails — check for extreme observations |
| **Durbin-Watson** | Autocorrelation of residuals (lag 1) | **Between 1.5 and 2.5** (2.0 = no autocorrelation) | Below 1.5 → positive autocorrelation; above 2.5 → negative autocorrelation (common with time-series data — may need lagged terms or ARIMA) |
| **Jarque-Bera (JB)** | Normality test using skewness + kurtosis | Probability (JB) **> 0.05** (statistic **close to 0**) | Residuals deviate from normality — predictions and confidence intervals may be unreliable |
| **Condition Number** | Multicollinearity among predictors | **Below 30** (roughly **< 1000** acceptable) | High condition number → predictors are highly correlated — remove or combine redundant features |


The prediction we made is into the future, we don't know how accurate the model in reality. To validate the model we run a train-test split to check for the accuracy of the model. The R2 that we have is only on the train data, we need to valid on a set of test cases as well. 

In [0]:
from sklearn.model_selection import train_test_split

train_x,test_x,train_y,test_y= train_test_split(X,y,test_size=0.2,random_state=42)

In [0]:
slr_model_1 = sm.OLS(train_y,train_x).fit()
print(slr_model_1.summary())

In [0]:
import numpy as np

In [0]:
pred_y = slr_model_1.predict(test_x[train_x.columns])

In [0]:
from sklearn import metrics
np.round(metrics.mean_squared_error(pred_y,test_y),2)

In [0]:
np.round(metrics.r2_score(pred_y,test_y),2)

In [0]:
plt.scatter(test_x["Year"],test_y,alpha=0.7)
plt.plot(test_x["Year"],pred_y,color="red",linewidth=2,label="SLR Test fit")
plt.legend()
plt.xlabel("Year")
plt.ylabel("Yield (bu/acre)")
plt.title("Oats Yield vs. Year")
plt.grid(True)
plt.show()

#Tracking Experiments Using MLflow

## `.fit()` vs `.fit_regularized()` for OLS (statsmodels)

### `.fit()` — Ordinary Least Squares (default)

Performs classic OLS estimation by minimizing the **residual sum of squares (RSS)**:

$$\hat{\beta} = \arg\min_{\beta} \sum_{i}(y_i - X_i\beta)^2$$

Key parameters (`OLS(...).fit(...)`):

| Parameter | Default | What it does |
|---|---|---|
| `method` | `'pinv'` | Solver for the normal equations: `'pinv'` (Moore–Penrose pseudo-inverse, robust to rank-deficiency), `'qr'` (QR decomposition), or `'lstsq'` (scipy least-squares). For OLS all give identical estimates on full-rank data — the choice is about numerical stability. |
| `cov_type` | `'nonrobust'` | Standard-error estimator: `'HC0'`–`'HC3'` (heteroskedasticity-consistent / White), `'HAC'` (Newey–West, heteroskedasticity + autocorrelation consistent, useful for this time-series data), `'cluster'` (cluster-robust). |
| `cov_kwds` | `None` | Extra options for `cov_type`, e.g. `{'maxlags': 5}` for HAC or `{'groups': ...}` for clustered. |
| `use_t` | varies | Use t-distribution instead of normal for p-values when a robust covariance is selected. |
| `disp` | `True` | Whether to show convergence output (mostly for non-OLS models). |

What the returned `RegressionResults` object gives you:
- `params` — estimated coefficients (e.g. the slope ≈ 0.9476 and intercept ≈ −1778 we saw above)
- `rsquared`, `rsquared_adj`, `fvalue`, `aic`, `bic`
- `bse`, `tvalues`, `pvalues`, `conf_int()` — inference that is **exact under the classical linear-model assumptions** (Gauss–Markov: residuals homoskedastic, uncorrelated, normal)
- Rich diagnostics: `Durbin-Watson`, `Omnibus`, `Jarque-Bera`, condition number (the table above)
- `predict(X_new)` for out-of-sample forecasts

> Because `.fit()` is unpenalized, the condition number warning on `Year` (large values like 1866–2027) inflates numerical sensitivity — scaling `Year` (e.g. `Year - 1866`) helps without changing predictions.

### `.fit_regularized()` — Penalized (Regularized) Estimation

Minimizes RSS **plus a penalty** on coefficient size:

$$\hat{\beta} = \arg\min_{\beta} \sum_i(y_i - X_i\beta)^2 + \alpha\big[(1-\lambda)\|\beta\|_2^2 + \lambda\|\beta\|_1\big]$$

Key parameters:

| Parameter | Default | What it does |
|---|---|---|
| `method` | `'elastic_net'` | Penalty family. `'elastic_net'` blends Ridge (L2) and Lasso (L1); `'sqrt_lasso'` uses the square-root Lasso objective, which adapts to heteroskedastic noise automatically (nice for agricultural yield data with changing variance over decades). |
| `alpha` | `0.0` | Overall penalty strength. `alpha = 0` → identical to plain OLS. Larger → more shrinkage (and L1 sparsity if `L1_wt > 0`). |
| `L1_wt` | `1.0` | The blend `λ` between L1 (Lasso, variable **selection** — can zero out coefficients) and L2 (Ridge, **shrinkage** — handles multicollinearity by spreading weight across correlated predictors). `L1_wt=0` → pure Ridge; `L1_wt=1` → pure Lasso. |
| `start_params` | `None` | Warm-start coefficients for the optimizer. |
| `refit` | `False` | If `True`, refits an **unpenalized** model using only the variables that survived the L1 zeroing — a built-in feature-selection pipeline. |
| `maxiter`, `tol`, `cnvrg_tol` | — | Optimizer controls for the coordinate-descent / fitting routine (max iterations, convergence tolerance). |

The returned `RegularizedResults` object is **much leaner**:
- `params` — penalized coefficient estimates
- `predict(X_new)` — predictions
- ✗ No `rsquared`, `fvalue`, `bse`, `tvalues`, `pvalues`, or `conf_int()` — **standard inference is not statistically valid under penalization** (the penalty biases the estimates, so classical SEs / t-tests no longer have their usual meaning). Evaluate regularized models with cross-validated prediction error (like the MSE / R² we computed on the test split) instead of the summary table.

### When to use which in this notebook

| Situation | Choice |
|---|---|
| Small model, few predictors, inference matters ("does Year significantly affect Yield?", confidence intervals) | `.fit()` |
| Heavy multicollinearity among many predictors | `.fit_regularized(method='elastic_net', L1_wt=0)` (Ridge) |
| Many irrelevant predictors — want automatic variable selection | `.fit_regularized(method='elastic_net', L1_wt=1)` (Lasso) |
| Heteroskedastic residuals (check the Omnibus/JB tests above) | `.fit(method='pinv', cov_type='HC3')` or `sqrt_lasso` |
| Pure prediction quality | Either — compare via train/test MSE, not the OLS summary |

For our single-predictor `Yield ~ Year` model, plain `.fit()` is appropriate — with one predictor there is no multicollinearity to fix and no variable selection to perform; regularization would only shrink the slope toward 0 and bias the 2027 forecast.


In [0]:
import itertools
import mlflow
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson

# One MLflow experiment; every combination of changeable .OLS() / .fit() attributes becomes a child run
experiment_name = "/Workspace/Users/doctornikhildjonathan@gmail.com/Exp_Hands-On-Business-Analytics-Zero2Hero/ols_attributes_grid"
mlflow.set_experiment(experiment_name)

# --- Attributes changeable in the sm.OLS(...) constructor ---

ols_param_grid = [
    {"missing": "none",  "hasconst": True},
    {"missing": "drop",  "hasconst": True},
    {"missing": "none",  "hasconst": False},
]

# --- Attributes changeable in .fit(...) ---

fit_param_grid = [
    {"method": "pinv",  "cov_type": "nonrobust"},
    {"method": "qr",    "cov_type": "nonrobust"},
    {"method": "lstsq", "cov_type": "nonrobust"},
    {"method": "pinv",  "cov_type": "HC0"},
    {"method": "pinv",  "cov_type": "HC1"},
    {"method": "pinv",  "cov_type": "HC2"},
    {"method": "pinv",  "cov_type": "HC3"},
    {"method": "pinv",  "cov_type": "HAC",      "cov_kwds": {"maxlags": 5}},
    {"method": "pinv",  "cov_type": "cluster",  "cov_kwds": {"groups": (df_oats["Year"] // 10 * 10).values}},
]

with mlflow.start_run(run_name="ols_attributes_grid_parent") as parent_run:
    mlflow.set_tags({"model": "statsmodels.OLS", "dataset": "Oats_Yield_1866"})
    mlflow.log_param("n_child_runs", len(ols_param_grid) * len(fit_param_grid))

    for ols_kwargs, fit_kwargs in itertools.product(ols_param_grid, fit_param_grid):
        run_name = (
            f"OLS-missing={ols_kwargs['missing']}-hasconst={ols_kwargs['hasconst']}-"
            f"fit-method={fit_kwargs['method']}-cov={fit_kwargs['cov_type']}"
        )
        with mlflow.start_run(run_name=run_name, nested=True) as child_run:
            try:
                # Constructor attributes
                model = sm.OLS(y, X, **ols_kwargs)
                # Fit attributes
                result = model.fit(**fit_kwargs)

                # Log all constructor + fit attributes as parameters
                for k, v in ols_kwargs.items():
                    mlflow.log_param(k, v)
                for k, v in fit_kwargs.items():
                    if k == "cov_kwds":
                        mlflow.log_param(k, str(v))
                    else:
                        mlflow.log_param(k, v)

                # Log coefficients and diagnostics as metrics
                mlflow.log_metrics({
                    "intercept": float(result.params.iloc[0]),
                    "slope_year": float(result.params.iloc[1]),
                    "intercept_pvalue": float(result.pvalues.iloc[0]),
                    "slope_pvalue": float(result.pvalues.iloc[1]),
                    "r_squared": result.rsquared,
                    "adj_r_squared": result.rsquared_adj,
                    "aic": result.aic,
                    "bic": result.bic,
                    "f_statistic": result.fvalue,
                    "durbin_watson": durbin_watson(result.resid),
                    "condition_number": float(np.linalg.cond(X)),
                })
                mlflow.set_tag("status", "success")
            except Exception as e:
                mlflow.set_tag("status", "failed")
                mlflow.set_tag("error", str(e))

print(f"Experiment: {experiment_name}")
print(f"Parent run: {parent_run.info.run_id} with {len(ols_param_grid) * len(fit_param_grid)} child runs")
print(f"Open the experiment in the Experiments sidebar to compare child runs side by side.")